# CATIVE Interactive Predictor 🚀

Select a startup from the dropdown below to instantly load its details and pass it through our trained Phase 1 and Phase 3 models. 
Watch how the **GMM Entropy** changes on ambiguous startups!

In [1]:
import pandas as pd
import numpy as np
import pickle
import ipywidgets as widgets
from IPython.display import display, HTML

# 1. Load Data
df = pd.read_csv('../data/df_processed.csv')
X_full = pd.read_csv('../data/X_full.csv').values

# 2. Load Models
with open('../outputs/models/svm_model.pkl', 'rb') as f: svm = pickle.load(f)
with open('../outputs/models/xgboost_model.pkl', 'rb') as f: xgb = pickle.load(f)
with open('../outputs/models/label_encoder.pkl', 'rb') as f: le = pickle.load(f)
with open('../outputs/models/pca_gmm.pkl', 'rb') as f: pca = pickle.load(f)
with open('../outputs/models/gmm_model.pkl', 'rb') as f: gmm = pickle.load(f)

# 3. Create Dropdown Options
options = []
for i, row in df.iterrows():
    options.append((f"#{i+1} | {row['sector']} | Funding: ${row['total_funding_usd']:,.0f}", i))

dropdown = widgets.Dropdown(
    options=options,
    value=0,
    description='Startup:',
    layout={'width': '500px'}
)

out = widgets.Output()

def on_change(change):
    with out:
        out.clear_output()
        idx = change['new']
        row = df.iloc[idx]
        
        # Get features and predict
        x = X_full[idx].reshape(1, -1)
        svm_pred = le.inverse_transform([svm.predict(x)[0]])[0]
        xgb_pred = le.inverse_transform([xgb.predict(x)[0]])[0]
        
        # GMM Entropy
        x_pca = pca.transform(x)
        probs = gmm.predict_proba(x_pca)[0]
        entropy = -np.sum(probs * np.log(probs + 1e-9))
        
        # Colors
        def get_color(val):
            if val == 'Emerging': return '#E07B54' # Orange
            if val == 'Growing': return '#5B8DB8' # Blue
            return '#4CAF7D' # Green
            
        ent_color = '#E07B54' if entropy > 0.8 else '#4CAF7D'
        
        html = f"""
        <div style="font-family: sans-serif; padding: 20px; border: 1px solid #ddd; border-radius: 8px; margin-top: 20px; max-width: 600px;">
            <h3>📊 Startup Raw Data</h3>
            <p><b>Sector:</b> {row['sector']} | <b>HQ:</b> {row['hq_city']}</p>
            <p><b>Hiring Roles:</b> {row['top_hiring_roles']}</p>
            <p><b>Funding:</b> ${row['total_funding_usd']:,.0f} | <b>Employees:</b> {row['employee_count']}</p>
            <p><b>Culture:</b> {row['culture_rating']} | <b>WLB:</b> {row['wlb_rating']} | <b>Salary:</b> {row['salary_rating']}</p>
            
            <hr style="margin: 20px 0;">
            
            <h3>🤖 Live Model Predictions</h3>
            <p style="font-size: 16px;"><b>Ground Truth:</b> <span style="color: {get_color(row['hire_quality_label'])}; font-weight: bold;">{row['hire_quality_label']}</span></p>
            <p style="font-size: 16px;"><b>SVM (Phase 1):</b> <span style="color: {get_color(svm_pred)}; font-weight: bold;">{svm_pred}</span></p>
            <p style="font-size: 16px;"><b>XGBoost (Phase 1):</b> <span style="color: {get_color(xgb_pred)}; font-weight: bold;">{xgb_pred}</span></p>
            <p style="font-size: 16px;"><b>GMM Entropy (Phase 3 Gate):</b> <span style="color: {ent_color}; font-weight: bold;">{entropy:.3f}</span> (Max 1.09)</p>
        </div>
        """
        display(HTML(html))

dropdown.observe(on_change, names='value')
display(dropdown, out)
# Trigger initial render
on_change({'new': 0})


Dropdown(description='Startup:', layout=Layout(width='500px'), options=(('#1 | Food Delivery | Funding: $14,00…

Output()